In [5]:
import pandas as pd
import numpy as np

import os
from scipy.stats import spearmanr, pearsonr, ttest_ind, percentileofscore
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, mean_squared_error, roc_curve, auc, confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.metrics import precision_score, recall_score
import itertools

In [17]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}
scores = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC Score"]

def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def magnitude_norm(data, data_min=0):
    data_max = np.max(np.abs(data))
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def get_significance_mark(p_value, alpha=0.05):
    if p_value < 0.001:
        return '***'
    elif p_value < 0.01:
        return '**'
    elif p_value < alpha:
        return '*'
    else:
        return 'n.s.'

In [7]:
data_dir = "data"
task_type = "classification"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
info_df = pd.read_csv(os.path.join(root_dir, data_dir, "input_info_VRC01_IC80.csv"))
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
seq_no_array = info_df['Seq_no'].values
true_labels = info_df["Label"].values
N_SEQ, L_SEQ = aligned_sequences.shape
print(aligned_sequences.shape)
print(np.unique(true_labels, return_counts=True))

resno_df = pd.read_csv(os.path.join(root_dir, data_dir, "selected_residues_IC80.csv"))
resno_array = resno_df["ResLabel"].values

vrc01_footprint = set([97, 123, 124, 198, 276, 278, 279, 280, 281, 282, 365, 366, 367, 368, 371, 427, 428, 429, 430, 455, 456, 457, 458, 459, 460, 461, 463, 465, 466, 467, 469, 472, 473, 474, 476])
vrc01_covary = set([46, 132, 138, 144, 150, 179, 181, 186, 190, 290, 321, 328, 354, 389, 394, 396, 397, 406])
vrc01_pngs = set([156, 157, 158, 229, 230, 231, 234, 235, 236, 616, 617, 618, 824, 825, 826])

vrc01_contact_residues = list(vrc01_footprint | vrc01_covary | vrc01_pngs)
vrc01_contact_residues = [str(_) for _ in vrc01_contact_residues]
mask_vrc01_contact_residues = np.isin(resno_array, vrc01_contact_residues)

res_of_interest = {
    "276": ["N"],
    "279": ["K", "S", "G"],
    "281": ["T", "S"]
    }

(889, 209)
(array([0, 1]), array([583, 306]))


In [8]:
def build_typeaware_arrays(sequences, array_scalar):
    """Expand attention and attribution arrays into amino acid-specific channels."""
    N_SEQ, L_SEQ = sequences.shape
    
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(sequences)
    array = np.zeros((N_SEQ, L_SEQ, N_AA))
    seq_idx = np.arange(N_SEQ)[:, None]
    pos_idx = np.arange(L_SEQ)[None, :]
    array[seq_idx, pos_idx, aa_idx_array] = array_scalar

    return array

def get_aa_counts(selected_aligned_seq):
    AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
    N_AA = len(AA_LIST)
    aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
    idx_to_aa = {v: k for k, v in aa_to_idx.items()}
    aa_idx_array = np.vectorize(aa_to_idx.get, otypes=[int])(selected_aligned_seq)
    alignments = np.zeros([aa_idx_array.shape[1], N_AA])
    for i in range(aa_idx_array.shape[1]):
        unique, counts = np.unique(aa_idx_array[:,i], return_counts=True)
        alignments[i, unique] = counts
    return alignments

def get_res_significance(scores, aligned_sequences, resno_array=resno_array, idx_to_aa=idx_to_aa):
    type_aware_scores = build_typeaware_arrays(aligned_sequences, scores)
    result_list = []
    for res_idx in range(L_SEQ):
        res_scores = type_aware_scores[:,res_idx]
        for aa_idx_1 in range(N_AA):
            res_aa_score = res_scores[:, aa_idx_1]
            score_1 = res_aa_score[res_aa_score != 0]
            if len(score_1) == 0:
                continue
            
            other_aa_indices = [k for k in idx_to_aa.keys() if k != aa_idx_1]
            not_res_aa_score_2d = res_scores[:, other_aa_indices]
            score_2 = not_res_aa_score_2d[not_res_aa_score_2d != 0].flatten()
            add_zeros = len(set(np.where((not_res_aa_score_2d == 0).all(axis=1))[0]) - set(np.where(res_aa_score != 0)[0]))
            if add_zeros > 0:
                score_2 = np.concatenate([score_2, np.zeros([add_zeros])]).flatten()

            assert len(score_1) + len(score_2) == len(aligned_sequences)

            if len(score_2) == 0:
                result_list.append({
                    "ResLabel": resno_array[res_idx],
                    "AA": idx_to_aa[aa_idx_1],
                    "AA_count": len(score_1),
                    "AA_mean_score": np.mean(score_1),
                    "AA_std_score": np.std(score_1),
                    "nonAA_count": 0,
                    "notAA_mean_score": 0,
                    "notAA_std_score": 0,
                    "pvalue": np.nan,
                })
                continue
            if len(score_1) > 1 and len(score_2) > 1 and np.var(score_1) > 1e-8 and np.var(score_2) > 1e-8:
                _, p_value = ttest_ind(score_1, score_2, equal_var=False)
            else:
                p_value = np.nan
            result_list.append({
                "ResLabel": resno_array[res_idx],
                "AA": idx_to_aa[aa_idx_1],
                "AA_count": len(score_1),
                "AA_mean_score": np.mean(score_1),
                "AA_std_score": np.std(score_1),
                "nonAA_count": len(score_2),
                "notAA_mean_score": np.mean(score_2),
                "notAA_std_score": np.std(score_2),
                "pvalue": p_value,
            })
    return pd.DataFrame(result_list)

def analyze_residues(feature_array, feature_name, model_id, aligned_seqs):
    """Calculates safe statistics for specific residues and returns a list of dictionaries."""
    records = []
    type_aware_scores = build_typeaware_arrays(aligned_seqs, feature_array)
    
    for resno, target_aas in res_of_interest.items():
        res_idx = np.where(resno_array == resno)[0][0]
        res_scores = type_aware_scores[:, res_idx]
        
        aa_idx = [index for index, value in enumerate(AA_LIST) if value in target_aas]
        res_aa_score = res_scores[:, aa_idx]
        score_1 = res_aa_score[res_aa_score != 0].flatten()

        other_aa_indices = [k for k in idx_to_aa.keys() if k not in aa_idx]
        not_res_aa_score_2d = res_scores[:, other_aa_indices]
        score_2 = not_res_aa_score_2d[not_res_aa_score_2d != 0].flatten()
        
        add_zeros = len(set(np.where((not_res_aa_score_2d == 0).all(axis=1))[0]) - set(np.where(res_aa_score != 0)[0]))
        if add_zeros > 0:
            score_2 = np.concatenate([score_2, np.zeros([add_zeros])]).flatten()
        
        assert len(score_1) + len(score_2) == len(aligned_seqs), f"Length mismatch at residue {resno}"

        # Safe Statistical Calculations
        n1, n2 = len(score_1), len(score_2)
        mean1 = np.mean(score_1) if n1 > 0 else np.nan
        std1 = np.std(score_1) if n1 > 0 else np.nan
        mean2 = np.mean(score_2) if n2 > 0 else np.nan
        std2 = np.std(score_2) if n2 > 0 else np.nan

        # Variance check to prevent catastrophic cancellation warnings
        if n1 > 1 and n2 > 1 and np.var(score_1) > 1e-8 and np.var(score_2) > 1e-8:
            _, p_val = ttest_ind(score_1, score_2, equal_var=False)
        else:
            p_val = np.nan

        records.append({
            "Model_ID": model_id,
            "Feature": feature_name,
            "Residue": resno,
            "Target_AAs": "/".join(target_aas),
            "Target_Mean": mean1,
            "Target_Std": std1,
            "Target_N": n1,
            "Other_Mean": mean2,
            "Other_Std": std2,
            "Other_N": n2,
            "P_Value": p_val
        })
    return records

In [33]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/pl/active/rdi_data/fahsai/PLM"
pred_dir = os.path.join(root_dir, "results/predictions/classification")
attr_dir = os.path.join(root_dir, "results/attribution_maps/classification")
attn_dir = os.path.join(root_dir, "results/attention_maps/classification/full")

print(f"Model Type: Fine-tuned")

model_metrics_records = []
residue_metrics_records = []
attr_metrics_records = []

for model_idx, model_id in enumerate(selected_models):
    print(f"Processing {model_id}...")
    predict_df = pd.read_csv(os.path.join(pred_dir, f"train_{model_id}.csv"))
    true_labels = predict_df["Label"].values
    pred_labels = []
    pos_pred_probabilities = []
    
    for pred in predict_df["Prediction"]:
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        pos_pred_probabilities.append(pred_1)
        pred_labels.append(0 if pred_0 > pred_1 else 1)

    pred_labels = np.array(pred_labels)
    assert len(true_labels) == len(pred_labels)
    all_attr_array_0 = np.zeros([N_SEQ, L_SEQ])
    all_attr_array_1 = np.zeros([N_SEQ, L_SEQ])
    cls_attn_scalar_array = np.zeros([N_SEQ, L_SEQ])

    for seq_idx, no in enumerate(info_df['Seq_no']):
        attr_0 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_0.npy"))
        all_attr_array_0[seq_idx] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_1.npy"))
        all_attr_array_1[seq_idx] = magnitude_norm(attr_1)

        attn = np.load(os.path.join(attn_dir, model_id, f"{no}_attentions.npy"))
        cls_attn_scalar_array[seq_idx] = min_max_norm(attn[0, 1:])

    # Global VRC01 Attention Analysis
    vrc01_samples = np.sum(cls_attn_scalar_array[:, mask_vrc01_contact_residues], axis=1) / mask_vrc01_contact_residues.sum()
    non_vrc01_samples = np.sum(cls_attn_scalar_array[:, ~mask_vrc01_contact_residues], axis=1) / (~mask_vrc01_contact_residues).sum()
    
    if np.var(vrc01_samples) > 1e-8 and np.var(non_vrc01_samples) > 1e-8:
        _, vrc01_p_value = ttest_ind(vrc01_samples, non_vrc01_samples, equal_var=False)
    else:
        vrc01_p_value = np.nan

    model_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1 Score": f1_score(true_labels, pred_labels, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, pos_pred_probabilities),
        "CLS-Attn_VRC01_mean": vrc01_samples.mean(),
        "CLS-Attn_VRC01_std": vrc01_samples.std(),
        "CLS-Attn_nonVRC01_mean": non_vrc01_samples.mean(),
        "CLS-Attn_nonVRC01_std": non_vrc01_samples.std(),
        "CLS-Attn_VRC01_pvalue": vrc01_p_value,
    })

    # Resolve Attribution Conflicts
    attr_1_array = all_attr_array_1.copy()
    attr_0_array = all_attr_array_0.copy()

    pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
    neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
    conflict_mask = (pos_conflict_mask | neg_conflict_mask)
    align_mask = ~conflict_mask

    abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
    abs_attr_score_array[conflict_mask] = 0
    attr_score_array = abs_attr_score_array.copy()
    attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

    attr_pred = attr_score_array.sum(axis=1)
    attr_pred[attr_pred < 0] = 0
    attr_pred[attr_pred > 0] = 1
    attr_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, attr_pred),
        "Precision": precision_score(true_labels, attr_pred, zero_division=0),
        "Recall": recall_score(true_labels, attr_pred, zero_division=0),
        "F1 Score": f1_score(true_labels, attr_pred, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, attr_score_array.sum(axis=1)),
    })

    # Filter for Correct Predictions
    correct_mask = (true_labels == pred_labels)
    correct_seq_nos = predict_df[correct_mask]["Seq_no"].values
    data_df = info_df.loc[info_df['Seq_no'].isin(correct_seq_nos)].copy()
    aligned_sequences_correct = np.array([list(seq) for seq in data_df['Sequence']])

    # Global Feature Significance Mapping
    cls_sig_df = get_res_significance(cls_attn_scalar_array[correct_mask], aligned_sequences_correct)
    cls_sig_df = cls_sig_df.merge(resno_df, on=["ResLabel"])

    attr_sig_df = get_res_significance(attr_score_array[correct_mask], aligned_sequences_correct)
    attr_sig_df = attr_sig_df.merge(resno_df, on=["ResLabel"])

    cls_subset = cls_sig_df[["ResLabel", "AA", "AA_mean_score", "AA_std_score", "notAA_mean_score", "notAA_std_score", "pvalue"]]
    attr_with_attn = attr_sig_df.merge(cls_subset, on=["ResLabel", "AA"], suffixes=("_attr", "_attn"))
    attr_with_attn.to_csv(os.path.join(pred_dir, f"attr_with_attn_{model_id}.csv"), index=False)

    # Residue-Specific Analysis (Stores data instead of printing)
    attr_records = analyze_residues(attr_score_array[correct_mask], "Attribution", model_id, aligned_sequences_correct)
    attn_records = analyze_residues(cls_attn_scalar_array[correct_mask], "CLS_Attention", model_id, aligned_sequences_correct)
    
    residue_metrics_records.extend(attr_records)
    residue_metrics_records.extend(attn_records)

# --- Final DataFrame Generation & Saving ---

# Model Performance DataFrame
model_metrics_df = pd.DataFrame(model_metrics_records)
model_metrics_df.to_csv(os.path.join(pred_dir, "performance_summary.csv"), index=False)

# Attr Performance DataFrame
attr_metrics_df = pd.DataFrame(attr_metrics_records)
attr_metrics_df.to_csv(os.path.join(pred_dir, "attr_performance_summary.csv"), index=False)

# Site-Specific Mutation DataFrame
residue_metrics_df = pd.DataFrame(residue_metrics_records)
residue_metrics_df.to_csv(os.path.join(pred_dir, "residue_specific_analysis.csv"), index=False)

print("--- Pipeline Complete ---")
print("Model Performance Summary:")
for score in scores:
    mean_score = model_metrics_df[score].mean()
    std_score = model_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

print("\nAttribution Performance Summary:")
for score in scores:
    mean_score = attr_metrics_df[score].mean()
    std_score = attr_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

Model Type: Fine-tuned
Processing rep_5...
Processing rep_6...
Processing rep_7...
Processing rep_9...
Processing rep_10...
--- Pipeline Complete ---
Model Performance Summary:
Accuracy = 0.948 +/- 0.006
Precision = 0.938 +/- 0.007
Recall = 0.908 +/- 0.014
F1 Score = 0.923 +/- 0.009
AUC Score = 0.982 +/- 0.003

Attribution Performance Summary:
Accuracy = 0.946 +/- 0.006
Precision = 0.930 +/- 0.020
Recall = 0.914 +/- 0.026
F1 Score = 0.921 +/- 0.009
AUC Score = 0.970 +/- 0.006


In [34]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
exp_type = "random"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = os.path.join(root_dir, "results", exp_type, "attribution")
attn_dir = os.path.join(root_dir, "results", exp_type, "attention")
pred_dir = os.path.join(root_dir, "results", exp_type, "predictions")

print(f"Model Type: {exp_type}")

model_metrics_records = []
residue_metrics_records = []

for model_idx, model_id in enumerate(selected_models):
    print(f"Processing {model_id}...")
    predict_df = pd.read_csv(os.path.join(pred_dir, f"predict_{model_id}.csv"))
    true_labels = predict_df["Label"].values
    pred_labels = []
    pos_pred_probabilities = []
    
    for pred in predict_df["Prediction"]:
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        pos_pred_probabilities.append(pred_1)
        pred_labels.append(0 if pred_0 > pred_1 else 1)

    pred_labels = np.array(pred_labels)
    assert len(true_labels) == len(pred_labels)
    all_attr_array_0 = np.zeros([N_SEQ, L_SEQ])
    all_attr_array_1 = np.zeros([N_SEQ, L_SEQ])
    cls_attn_scalar_array = np.zeros([N_SEQ, L_SEQ])

    for seq_idx, no in enumerate(info_df['Seq_no']):
        attr_0 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_0.npy"))
        all_attr_array_0[seq_idx] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_1.npy"))
        all_attr_array_1[seq_idx] = magnitude_norm(attr_1)

        attn = np.load(os.path.join(attn_dir, model_id, f"{no}_attentions.npy"))
        cls_attn_scalar_array[seq_idx] = min_max_norm(attn[0, 1:])

    # Global VRC01 Attention Analysis
    vrc01_samples = np.sum(cls_attn_scalar_array[:, mask_vrc01_contact_residues], axis=1) / mask_vrc01_contact_residues.sum()
    non_vrc01_samples = np.sum(cls_attn_scalar_array[:, ~mask_vrc01_contact_residues], axis=1) / (~mask_vrc01_contact_residues).sum()
    
    if np.var(vrc01_samples) > 1e-8 and np.var(non_vrc01_samples) > 1e-8:
        _, vrc01_p_value = ttest_ind(vrc01_samples, non_vrc01_samples, equal_var=False)
    else:
        vrc01_p_value = np.nan

    model_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1 Score": f1_score(true_labels, pred_labels, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, pos_pred_probabilities),
        "CLS-Attn_VRC01_mean": vrc01_samples.mean(),
        "CLS-Attn_VRC01_std": vrc01_samples.std(),
        "CLS-Attn_nonVRC01_mean": non_vrc01_samples.mean(),
        "CLS-Attn_nonVRC01_std": non_vrc01_samples.std(),
        "CLS-Attn_VRC01_pvalue": vrc01_p_value,
    })

    # Resolve Attribution Conflicts
    attr_1_array = all_attr_array_1.copy()
    attr_0_array = all_attr_array_0.copy()

    pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
    neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
    conflict_mask = (pos_conflict_mask | neg_conflict_mask)
    align_mask = ~conflict_mask

    abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
    abs_attr_score_array[conflict_mask] = 0
    attr_score_array = abs_attr_score_array.copy()
    attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

    attr_pred = attr_score_array.sum(axis=1)
    attr_pred[attr_pred < 0] = 0
    attr_pred[attr_pred > 0] = 1
    attr_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, attr_pred),
        "Precision": precision_score(true_labels, attr_pred, zero_division=0),
        "Recall": recall_score(true_labels, attr_pred, zero_division=0),
        "F1 Score": f1_score(true_labels, attr_pred, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, attr_score_array.sum(axis=1)),
    })

    # Filter for Correct Predictions
    correct_mask = (true_labels == pred_labels)
    correct_seq_nos = predict_df[correct_mask]["Seq_no"].values
    data_df = info_df.loc[info_df['Seq_no'].isin(correct_seq_nos)].copy()
    aligned_sequences_correct = np.array([list(seq) for seq in data_df['Sequence']])

    # Global Feature Significance Mapping
    cls_sig_df = get_res_significance(cls_attn_scalar_array[correct_mask], aligned_sequences_correct)
    cls_sig_df = cls_sig_df.merge(resno_df, on=["ResLabel"])

    attr_sig_df = get_res_significance(attr_score_array[correct_mask], aligned_sequences_correct)
    attr_sig_df = attr_sig_df.merge(resno_df, on=["ResLabel"])

    cls_subset = cls_sig_df[["ResLabel", "AA", "AA_mean_score", "AA_std_score", "notAA_mean_score", "notAA_std_score", "pvalue"]]
    attr_with_attn = attr_sig_df.merge(cls_subset, on=["ResLabel", "AA"], suffixes=("_attr", "_attn"))
    attr_with_attn.to_csv(os.path.join(pred_dir, f"attr_with_attn_{model_id}.csv"), index=False)

    # Residue-Specific Analysis (Stores data instead of printing)
    attr_records = analyze_residues(attr_score_array[correct_mask], "Attribution", model_id, aligned_sequences_correct)
    attn_records = analyze_residues(cls_attn_scalar_array[correct_mask], "CLS_Attention", model_id, aligned_sequences_correct)
    
    residue_metrics_records.extend(attr_records)
    residue_metrics_records.extend(attn_records)

# --- Final DataFrame Generation & Saving ---

# Model Performance DataFrame
model_metrics_df = pd.DataFrame(model_metrics_records)
model_metrics_df.to_csv(os.path.join(pred_dir, "performance_summary.csv"), index=False)

# Attr Performance DataFrame
attr_metrics_df = pd.DataFrame(attr_metrics_records)
attr_metrics_df.to_csv(os.path.join(pred_dir, "attr_performance_summary.csv"), index=False)

# Site-Specific Mutation DataFrame
residue_metrics_df = pd.DataFrame(residue_metrics_records)
residue_metrics_df.to_csv(os.path.join(pred_dir, "residue_specific_analysis.csv"), index=False)

print("--- Pipeline Complete ---")
print("Model Performance Summary:")
for score in scores:
    mean_score = model_metrics_df[score].mean()
    std_score = model_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

print("\nAttribution Performance Summary:")
for score in scores:
    mean_score = attr_metrics_df[score].mean()
    std_score = attr_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

Model Type: random
Processing rep_5...
Processing rep_6...
Processing rep_7...
Processing rep_9...
Processing rep_10...
--- Pipeline Complete ---
Model Performance Summary:
Accuracy = 0.531 +/- 0.171
Precision = 0.138 +/- 0.188
Recall = 0.399 +/- 0.547
F1 Score = 0.205 +/- 0.280
AUC Score = 0.517 +/- 0.052

Attribution Performance Summary:
Accuracy = 0.742 +/- 0.237
Precision = 0.611 +/- 0.354
Recall = 0.660 +/- 0.420
F1 Score = 0.589 +/- 0.384
AUC Score = 0.743 +/- 0.241


In [40]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
exp_type = "freezedPLM"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = os.path.join(root_dir, "results", exp_type, "attribution")
attn_dir = os.path.join(root_dir, "results", exp_type, "attention")
pred_dir = os.path.join(root_dir, "results", exp_type, "predictions")

print(f"Model Type: {exp_type}")

model_metrics_records = []
residue_metrics_records = []

for model_idx, model_id in enumerate(selected_models):
    print(f"Processing {model_id}...")
    predict_df = pd.read_csv(os.path.join(pred_dir, f"predict_{model_id}.csv"))
    true_labels = predict_df["Label"].values
    pred_labels = []
    pos_pred_probabilities = []
    
    for pred in predict_df["Prediction"]:
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        pos_pred_probabilities.append(pred_1)
        pred_labels.append(0 if pred_0 > pred_1 else 1)

    pred_labels = np.array(pred_labels)
    assert len(true_labels) == len(pred_labels)
    all_attr_array_0 = np.zeros([N_SEQ, L_SEQ])
    all_attr_array_1 = np.zeros([N_SEQ, L_SEQ])
    cls_attn_scalar_array = np.zeros([N_SEQ, L_SEQ])

    for seq_idx, no in enumerate(info_df['Seq_no']):
        attr_0 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_0.npy"))
        all_attr_array_0[seq_idx] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_1.npy"))
        all_attr_array_1[seq_idx] = magnitude_norm(attr_1)

        # attn = np.load(os.path.join(attn_dir, model_id, f"{no}_attentions.npy"))
        # cls_attn_scalar_array[seq_idx] = min_max_norm(attn[0, 1:])

    # Global VRC01 Attention Analysis
    # vrc01_samples = np.sum(cls_attn_scalar_array[:, mask_vrc01_contact_residues], axis=1) / mask_vrc01_contact_residues.sum()
    # non_vrc01_samples = np.sum(cls_attn_scalar_array[:, ~mask_vrc01_contact_residues], axis=1) / (~mask_vrc01_contact_residues).sum()
    
    # if np.var(vrc01_samples) > 1e-8 and np.var(non_vrc01_samples) > 1e-8:
    #     _, vrc01_p_value = ttest_ind(vrc01_samples, non_vrc01_samples, equal_var=False)
    # else:
    #     vrc01_p_value = np.nan

    # model_metrics_records.append({
    #     "Model_ID": model_id,
    #     "Accuracy": accuracy_score(true_labels, pred_labels),
    #     "Precision": precision_score(true_labels, pred_labels, zero_division=0),
    #     "Recall": recall_score(true_labels, pred_labels, zero_division=0),
    #     "F1 Score": f1_score(true_labels, pred_labels, zero_division=0),
    #     "AUC Score": roc_auc_score(true_labels, pos_pred_probabilities),
    #     "CLS-Attn_VRC01_mean": vrc01_samples.mean(),
    #     "CLS-Attn_VRC01_std": vrc01_samples.std(),
    #     "CLS-Attn_nonVRC01_mean": non_vrc01_samples.mean(),
    #     "CLS-Attn_nonVRC01_std": non_vrc01_samples.std(),
    #     "CLS-Attn_VRC01_pvalue": vrc01_p_value,
    # })

    # Resolve Attribution Conflicts
    attr_1_array = all_attr_array_1.copy()
    attr_0_array = all_attr_array_0.copy()

    pos_conflict_mask = (attr_0_array > 0) & (attr_1_array > 0)
    neg_conflict_mask = (attr_0_array < 0) & (attr_1_array < 0)
    conflict_mask = (pos_conflict_mask | neg_conflict_mask)
    align_mask = ~conflict_mask

    abs_attr_score_array = (np.abs(attr_0_array) + np.abs(attr_1_array)) / 2
    abs_attr_score_array[conflict_mask] = 0
    attr_score_array = abs_attr_score_array.copy()
    attr_score_array[(align_mask & (attr_1_array < 0))] *= -1

    attr_pred = attr_score_array.sum(axis=1)
    attr_pred[attr_pred < 0] = 0
    attr_pred[attr_pred > 0] = 1
    attr_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, attr_pred),
        "Precision": precision_score(true_labels, attr_pred, zero_division=0),
        "Recall": recall_score(true_labels, attr_pred, zero_division=0),
        "F1 Score": f1_score(true_labels, attr_pred, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, attr_score_array.sum(axis=1)),
    })

    # Filter for Correct Predictions
    # correct_mask = (true_labels == pred_labels)
    # correct_seq_nos = predict_df[correct_mask]["Seq_no"].values
    # data_df = info_df.loc[info_df['Seq_no'].isin(correct_seq_nos)].copy()
    # aligned_sequences_correct = np.array([list(seq) for seq in data_df['Sequence']])

    # # Global Feature Significance Mapping
    # cls_sig_df = get_res_significance(cls_attn_scalar_array[correct_mask], aligned_sequences_correct)
    # cls_sig_df = cls_sig_df.merge(resno_df, on=["ResLabel"])

    # attr_sig_df = get_res_significance(attr_score_array[correct_mask], aligned_sequences_correct)
    # attr_sig_df = attr_sig_df.merge(resno_df, on=["ResLabel"])

    # cls_subset = cls_sig_df[["ResLabel", "AA", "AA_mean_score", "AA_std_score", "notAA_mean_score", "notAA_std_score", "pvalue"]]
    # attr_with_attn = attr_sig_df.merge(cls_subset, on=["ResLabel", "AA"], suffixes=("_attr", "_attn"))
    # attr_with_attn.to_csv(os.path.join(pred_dir, f"attr_with_attn_{model_id}.csv"), index=False)

    # # Residue-Specific Analysis (Stores data instead of printing)
    # attr_records = analyze_residues(attr_score_array[correct_mask], "Attribution", model_id, aligned_sequences_correct)
    # attn_records = analyze_residues(cls_attn_scalar_array[correct_mask], "CLS_Attention", model_id, aligned_sequences_correct)
    
    # residue_metrics_records.extend(attr_records)
    # residue_metrics_records.extend(attn_records)

# --- Final DataFrame Generation & Saving ---

# Model Performance DataFrame
# model_metrics_df = pd.DataFrame(model_metrics_records)
# model_metrics_df.to_csv(os.path.join(pred_dir, "performance_summary.csv"), index=False)
model_metrics_df = pd.read_csv(os.path.join(pred_dir, "performance_summary.csv"))

# Attr Performance DataFrame
attr_metrics_df = pd.DataFrame(attr_metrics_records)
attr_metrics_df.to_csv(os.path.join(pred_dir, "attr_performance_summary.csv"), index=False)

# Site-Specific Mutation DataFrame
# residue_metrics_df = pd.DataFrame(residue_metrics_records)
# residue_metrics_df.to_csv(os.path.join(pred_dir, "residue_specific_analysis.csv"), index=False)

print("--- Pipeline Complete ---")
print("Model Performance Summary:")
for score in scores:
    mean_score = model_metrics_df[score].mean()
    std_score = model_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

print("\nAttribution Performance Summary:")
for score in scores:
    mean_score = attr_metrics_df[score].mean()
    std_score = attr_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

Model Type: freezedPLM
Processing rep_5...
Processing rep_6...
Processing rep_7...
Processing rep_9...
Processing rep_10...
--- Pipeline Complete ---
Model Performance Summary:
Accuracy = 0.471 +/- 0.169
Precision = 0.206 +/- 0.188
Recall = 0.590 +/- 0.539
F1 Score = 0.305 +/- 0.279
AUC Score = 0.516 +/- 0.060

Attribution Performance Summary:
Accuracy = 0.609 +/- 0.194
Precision = 0.368 +/- 0.387
Recall = 0.417 +/- 0.480
F1 Score = 0.286 +/- 0.344
AUC Score = 0.556 +/- 0.175


In [36]:
selected_models = ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
exp_type = "cv" 
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
attr_dir = os.path.join(root_dir, "results", exp_type, "attribution")
attn_dir = os.path.join(root_dir, "results", exp_type, "attention")
pred_dir = os.path.join(root_dir, "results", exp_type)
sum_predict_df = pd.read_csv((os.path.join(root_dir, "results", exp_type, "predictions_summary_classification.csv")))

model_metrics_records = []
residue_metrics_records = []

for model_id in selected_models:
    print(f"Processing {model_id}...")
    fold_num = int(model_id[-1])
    predict_df = sum_predict_df[sum_predict_df["fold"] == fold_num]
    true_labels = predict_df["actual_label"].values
    pred_labels = []
    pos_pred_probabilities = []
    
    for pred in predict_df["prediction"]:
        pred_0 = float(pred.split()[1].split(",")[0])
        pred_1 = float(pred.split()[3].split("}")[0])
        pos_pred_probabilities.append(pred_1)
        pred_labels.append(0 if pred_0 > pred_1 else 1)

    pred_labels = np.array(pred_labels)
    assert len(true_labels) == len(pred_labels)

    all_attr_array_0 = np.zeros([N_SEQ, L_SEQ])
    all_attr_array_1 = np.zeros([N_SEQ, L_SEQ])
    cls_attn_scalar_array = np.zeros([N_SEQ, L_SEQ])
    assert (predict_df.Seq_no.values == info_df.Seq_no.values).all()
    for seq_idx, no in enumerate(info_df['Seq_no']):
        attr_0 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_0.npy"))
        all_attr_array_0[seq_idx] = magnitude_norm(attr_0)

        attr_1 = np.load(os.path.join(attr_dir, model_id, f"{no}_attribution_ig_class_1.npy"))
        all_attr_array_1[seq_idx] = magnitude_norm(attr_1)

        attn = np.load(os.path.join(attn_dir, model_id, f"{no}_attentions.npy"))
        cls_attn_scalar_array[seq_idx] = min_max_norm(attn[0, 1:])

    # Global VRC01 Attention Analysis
    vrc01_samples = np.sum(cls_attn_scalar_array[:, mask_vrc01_contact_residues], axis=1) / mask_vrc01_contact_residues.sum()
    non_vrc01_samples = np.sum(cls_attn_scalar_array[:, ~mask_vrc01_contact_residues], axis=1) / (~mask_vrc01_contact_residues).sum()
    
    if np.var(vrc01_samples) > 1e-8 and np.var(non_vrc01_samples) > 1e-8:
        _, vrc01_p_value = ttest_ind(vrc01_samples, non_vrc01_samples, equal_var=False)
    else:
        vrc01_p_value = np.nan

    model_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1 Score": f1_score(true_labels, pred_labels, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, pos_pred_probabilities),
        "CLS-Attn_VRC01_mean": vrc01_samples.mean(),
        "CLS-Attn_VRC01_std": vrc01_samples.std(),
        "CLS-Attn_nonVRC01_mean": non_vrc01_samples.mean(),
        "CLS-Attn_nonVRC01_std": non_vrc01_samples.std(),
        "CLS-Attn_VRC01_pvalue": vrc01_p_value,
    })

    attr_pred = attr_score_array.sum(axis=1)
    attr_pred[attr_pred < 0] = 0
    attr_pred[attr_pred > 0] = 1
    attr_metrics_records.append({
        "Model_ID": model_id,
        "Accuracy": accuracy_score(true_labels, attr_pred),
        "Precision": precision_score(true_labels, attr_pred, zero_division=0),
        "Recall": recall_score(true_labels, attr_pred, zero_division=0),
        "F1 Score": f1_score(true_labels, attr_pred, zero_division=0),
        "AUC Score": roc_auc_score(true_labels, attr_score_array.sum(axis=1)),
    })

    # Filter for Correct Predictions
    correct_mask = (true_labels == pred_labels)
    correct_seq_nos = predict_df[correct_mask]["Seq_no"].values
    data_df = info_df.loc[info_df['Seq_no'].isin(correct_seq_nos)].copy()
    aligned_sequences_correct = np.array([list(seq) for seq in data_df['Sequence']])

    # Global Feature Significance Mapping
    cls_sig_df = get_res_significance(cls_attn_scalar_array[correct_mask], aligned_sequences_correct)
    cls_sig_df = cls_sig_df.merge(resno_df, on=["ResLabel"])

    attr_sig_df = get_res_significance(attr_score_array[correct_mask], aligned_sequences_correct)
    attr_sig_df = attr_sig_df.merge(resno_df, on=["ResLabel"])

    cls_subset = cls_sig_df[["ResLabel", "AA", "AA_mean_score", "AA_std_score", "notAA_mean_score", "notAA_std_score", "pvalue"]]
    attr_with_attn = attr_sig_df.merge(cls_subset, on=["ResLabel", "AA"], suffixes=("_attr", "_attn"))
    attr_with_attn.to_csv(os.path.join(pred_dir, f"attr_with_attn_{model_id}.csv"), index=False)

    # Residue-Specific Analysis (Stores data instead of printing)
    attr_records = analyze_residues(attr_score_array[correct_mask], "Attribution", model_id, aligned_sequences_correct)
    attn_records = analyze_residues(cls_attn_scalar_array[correct_mask], "CLS_Attention", model_id, aligned_sequences_correct)
    
    residue_metrics_records.extend(attr_records)
    residue_metrics_records.extend(attn_records)

# --- Final DataFrame Generation & Saving ---

# Model Performance DataFrame
model_metrics_df = pd.DataFrame(model_metrics_records)
model_metrics_df.to_csv(os.path.join(pred_dir, "performance_summary.csv"), index=False)

# Attr Performance DataFrame
attr_metrics_df = pd.DataFrame(attr_metrics_records)
attr_metrics_df.to_csv(os.path.join(pred_dir, "attr_performance_summary.csv"), index=False)

# Site-Specific Mutation DataFrame
residue_metrics_df = pd.DataFrame(residue_metrics_records)
residue_metrics_df.to_csv(os.path.join(pred_dir, "residue_specific_analysis.csv"), index=False)

print("--- Pipeline Complete ---")
print("Model Performance Summary:")
for score in scores:
    mean_score = model_metrics_df[score].mean()
    std_score = model_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

print("\nAttribution Performance Summary:")
for score in scores:
    mean_score = attr_metrics_df[score].mean()
    std_score = attr_metrics_df[score].std()
    print(f"{score} = {mean_score:.3f} +/- {std_score:.3f}")

Processing fold_1...
Processing fold_2...
Processing fold_3...
Processing fold_4...
Processing fold_5...
--- Pipeline Complete ---
Model Performance Summary:
Accuracy = 0.870 +/- 0.021
Precision = 0.833 +/- 0.045
Recall = 0.780 +/- 0.028
F1 Score = 0.805 +/- 0.029
AUC Score = 0.920 +/- 0.014

Attribution Performance Summary:
Accuracy = 0.713 +/- 0.195
Precision = 0.408 +/- 0.412
Recall = 0.440 +/- 0.466
F1 Score = 0.393 +/- 0.421
AUC Score = 0.671 +/- 0.220


In [32]:
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
exp_type = "freezedPLM"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
pred_dir = "results/"+exp_type+"/predictions"
freezedPLM_df = pd.read_csv(os.path.join(root_dir, pred_dir, "residue_specific_analysis.csv"))

selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
exp_type = "random"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
pred_dir = "results/"+exp_type+"/predictions"
random_df = pd.read_csv(os.path.join(root_dir, pred_dir, "residue_specific_analysis.csv"))

selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
exp_type = "cv"
root_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis"
pred_dir = "results/"+exp_type
cv_df = pd.read_csv(os.path.join(root_dir, pred_dir, "residue_specific_analysis.csv"))

selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]
root_dir = "/pl/active/rdi_data/fahsai/PLM"
pred_dir = "results/predictions/classification"
full_df = pd.read_csv(os.path.join(root_dir, pred_dir, "residue_specific_analysis.csv"))

dict_dfs = {
    "random": random_df,
    "freezedPLM": freezedPLM_df,
    "cv": cv_df,
    "full": full_df
}

In [49]:
selected_resno = 279
selected_feature = "Attribution"
print(selected_feature, "for", selected_resno)
for exp_type, df in dict_dfs.items():
    print("---", exp_type, "---")
    selected_df = df[(df.Residue == selected_resno) & (df.Feature == selected_feature)]
    print(selected_df[["Model_ID", "Target_AAs", "Target_Mean", "Target_Std", "Other_Mean", "Other_Std", "P_Value"]].to_string(index=False))
    selected_df[["Model_ID", "Target_AAs", "Target_Mean", "Target_Std", "Other_Mean", "Other_Std", "P_Value"]].T.to_csv(exp_type+".csv")

Attribution for 279
--- random ---
Model_ID Target_AAs  Target_Mean  Target_Std  Other_Mean  Other_Std  P_Value
   rep_5      K/S/G    -0.028010    0.061444    0.011717   0.045392 0.344513
   rep_6      K/S/G          NaN         NaN    0.007138   0.017065      NaN
   rep_7      K/S/G     0.066976    0.024892    0.149827   0.069029 0.000005
   rep_9      K/S/G     0.012246    0.049433   -0.045143   0.045565 0.048368
  rep_10      K/S/G          NaN         NaN    0.030068   0.036280      NaN
--- freezedPLM ---
Model_ID Target_AAs  Target_Mean  Target_Std  Other_Mean  Other_Std  P_Value
   rep_5      K/S/G     0.036668    0.088155    0.005287   0.013674 0.343540
   rep_6      K/S/G          NaN         NaN   -0.108254   0.109016      NaN
   rep_7      K/S/G          NaN         NaN   -0.044348   0.074129      NaN
   rep_9      K/S/G    -0.078650    0.036857   -0.032245   0.047886 0.012424
  rep_10      K/S/G     0.108187    0.000000    0.053314   0.053453      NaN
--- cv ---
Model_ID Ta